# TP Integrador — Dataset Nursery

In [ ]:
# @title Imports y parámetros
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# carpeta raíz del proyecto (funciona sin importar desde dónde se ejecute el notebook)
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "datos.py").exists())
RUTA_DATOS = RAIZ / "data" / "raw" / "Datos.txt"

# para poder importar desde src/
sys.path.append(str(RAIZ / "src"))
from datos import cargar_datos
import AA_utils

SEMILLA = 42  # para que los resultados sean reproducibles

In [ ]:
# @title Cargar dataset
columnas= ['parents', 'has_nurs', 'form', 'children', 'housing', 'finance', 'social', 'health', 'clase']
df= pd.read_csv(RUTA_DATOS, header=None, names=columnas, dtype=str)

print(f"Cantidad de registros: {df.shape[0]}")
print(f"Cantidad de columnas: {df.shape[1]}")
print("\nValores nulos por columna:")
print(df.isnull().sum())

display(df.head())
display(df.describe())   # count / unique / top / freq de cada atributo (todos son nominales)

### Qué vemos en la carga

- Hay 12.960 registros y 9 columnas (8 atributos más `clase`), sin valores nulos.
- En los 8 atributos, la `freq` del valor más frecuente es exactamente 12.960 / `unique`: cada valor aparece la misma cantidad de veces.
- El producto de los `unique` de los atributos da 3·5·4·4·3·2·3·3 = 12.960, igual que la cantidad de registros. Coincide con la descripción del dataset: las instancias cubren todo el espacio de atributos, cada combinación una vez.
- Por eso ningún atributo está desbalanceado, y lo que hay que analizar es cómo se relaciona cada uno con la clase.
- `clase` es la única variable despareja: tiene 5 valores y `not_recom` aparece 4.320 veces, cuando un reparto parejo daría 2.592.

In [ ]:
pd.crosstab(df['health'], df['clase'])

### Qué vemos en el cruce de `health` con la clase

- Las 4.320 filas con `health = not_recom` son exactamente las 4.320 de la clase `not_recom`. Esa clase queda definida por una sola columna.
- Si comparáramos `not_recom` contra `very_recom`, cualquier modelo separaría todo con esa regla y no habría nada que comparar.
- `very_recom` solo aparece con `health = recommended`.
- Por eso elegimos `priority` como clase mayoritaria: no queda definida por un único atributo y comparte `health = recommended` con `very_recom` (2.412 filas de `priority` y 328 de `very_recom`).

In [ ]:
# @title Subconjunto binario: priority vs very_recom
df_bin = df[df['clase'].isin(['priority', 'very_recom'])].copy()

print(f"Cantidad de registros: {df_bin.shape[0]}")
print(df_bin['clase'].value_counts())
print()
print(f"Accuracy de predecir siempre priority: {(df_bin['clase'] == 'priority').mean():.3f}")

### Qué vemos en el subconjunto

- Quedan 4.594 registros: 4.266 de `priority` y 328 de `very_recom`, un desbalance de aproximadamente 13 a 1.
- Un modelo que siempre responda `priority` ya tendría 0,929 de accuracy, así que esa métrica sola no alcanza para evaluar los modelos.